# WGAN-GP for Fashion MNIST (Task)

**Task:** Build and train a WGAN-GP (Wasserstein GAN with Gradient Penalty) network for the Fashion MNIST dataset (`tf.keras.datasets.fashion_mnist`), choosing the latent space dimension. Use the **training** split of Fashion MNIST for GAN training. For a random set of points in the latent space, build and visualize the generated images.

## 1. Imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, optimizers, metrics

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

## 2. Load Fashion MNIST (training set only)

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.fashion_mnist.load_data()

# Use training set only for GAN training
x_train = x_train.astype("float32")
# Scale to [-1, 1] for generator tanh output
x_train = (x_train / 255.0) * 2.0 - 1.0
# Add channel dimension: (N, 28, 28) -> (N, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)

print("Training set shape:", x_train.shape)
print("Value range: [{:.2f}, {:.2f}]".format(x_train.min(), x_train.max()))

## 3. Hyperparameters (latent dimension and WGAN-GP)

In [ ]:
IMAGE_SIZE = 28
CHANNELS = 1
Z_DIM = 128          # Latent space dimension (common choice for 28x28)
BATCH_SIZE = 64
EPOCHS = 30
CRITIC_STEPS = 3     # Number of critic updates per generator step
GP_WEIGHT = 10.0     # Gradient penalty weight
LEARNING_RATE = 0.0002
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9

## 4. Build the Critic (no BatchNorm for WGAN-GP)

In [ ]:
critic_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same")(critic_input)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(512, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(1, kernel_size=2, strides=1, padding="valid")(x)
critic_output = layers.Flatten()(x)

critic = models.Model(critic_input, critic_output, name="critic")
critic.summary()

## 5. Build the Generator (28×28×1 output, tanh)

In [ ]:
generator_input = layers.Input(shape=(Z_DIM,))
x = layers.Dense(7 * 7 * 256, use_bias=False)(generator_input)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Reshape((7, 7, 256))(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
generator_output = layers.Conv2D(CHANNELS, kernel_size=4, strides=1, padding="same", activation="tanh")(x)

generator = models.Model(generator_input, generator_output, name="generator")
generator.summary()

## 6. WGAN-GP model (gradient penalty + train_step)

In [ ]:
class WGANGP(models.Model):
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super(WGANGP, self).__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def compile(self, c_optimizer, g_optimizer):
        super(WGANGP, self).compile()
        self.c_optimizer = c_optimizer
        self.g_optimizer = g_optimizer
        self.c_wass_metric = metrics.Mean(name="c_wass")
        self.c_gp_metric = metrics.Mean(name="c_gp")
        self.c_loss_metric = metrics.Mean(name="c_loss")
        self.g_loss_metric = metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.c_loss_metric, self.c_wass_metric, self.c_gp_metric, self.g_loss_metric]

    def gradient_penalty(self, batch_size, real_images, fake_images):
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
        interpolated = real_images + alpha * (fake_images - real_images)
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic(interpolated, training=True)
        grads = gp_tape.gradient(pred, interpolated)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        gp = tf.reduce_mean((norm - 1.0) ** 2)
        return gp

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]

        for _ in range(self.critic_steps):
            random_latent = tf.random.normal(shape=(batch_size, self.latent_dim))
            with tf.GradientTape() as tape:
                fake_images = self.generator(random_latent, training=True)
                fake_pred = self.critic(fake_images, training=True)
                real_pred = self.critic(real_images, training=True)
                c_wass = tf.reduce_mean(fake_pred) - tf.reduce_mean(real_pred)
                c_gp = self.gradient_penalty(batch_size, real_images, fake_images)
                c_loss = c_wass + self.gp_weight * c_gp
            grads = tape.gradient(c_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(zip(grads, self.critic.trainable_variables))

        random_latent = tf.random.normal(shape=(batch_size, self.latent_dim))
        with tf.GradientTape() as tape:
            fake_images = self.generator(random_latent, training=True)
            fake_pred = self.critic(fake_images, training=True)
            g_loss = -tf.reduce_mean(fake_pred)
        grads = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_variables))

        self.c_loss_metric.update_state(c_loss)
        self.c_wass_metric.update_state(c_wass)
        self.c_gp_metric.update_state(c_gp)
        self.g_loss_metric.update_state(g_loss)
        return {m.name: m.result() for m in self.metrics}


wgangp = WGANGP(
    critic=critic,
    generator=generator,
    latent_dim=Z_DIM,
    critic_steps=CRITIC_STEPS,
    gp_weight=GP_WEIGHT,
)
wgangp.compile(
    c_optimizer=optimizers.Adam(LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
    g_optimizer=optimizers.Adam(LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
)

## 7. Train the WGAN-GP on Fashion MNIST

In [ ]:
history = wgangp.fit(
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["c_loss"], label="critic")
plt.plot(history.history["g_loss"], label="generator")
plt.legend()
plt.title("WGAN-GP losses")
plt.subplot(1, 2, 2)
plt.plot(history.history["c_gp"], label="gradient penalty")
plt.legend()
plt.title("Gradient penalty")
plt.tight_layout()
plt.show()

## 8. Sample random points in latent space and visualize

In [ ]:
n_samples = 16
z_sample = np.random.normal(size=(n_samples, Z_DIM)).astype("float32")
generated = wgangp.generator.predict(z_sample, verbose=0)

n_cols = 4
n_rows = (n_samples + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
axes = axes.flatten()
for i in range(n_samples):
    img = generated[i].squeeze()
    img = (img + 1.0) / 2.0  # [-1,1] -> [0,1]
    axes[i].imshow(img, cmap="gray_r")
    axes[i].axis("off")
for j in range(n_samples, len(axes)):
    axes[j].axis("off")
plt.suptitle("Generated Fashion MNIST images (random latent points)")
plt.tight_layout()
plt.show()

## 9. Additional sample (different random seed)

In [ ]:
np.random.seed(42)
z_sample2 = np.random.normal(size=(10, Z_DIM)).astype("float32")
gen2 = wgangp.generator.predict(z_sample2, verbose=0)

fig2, ax2 = plt.subplots(2, 5, figsize=(10, 4))
for i in range(10):
    r, c = i // 5, i % 5
    img = (gen2[i].squeeze() + 1.0) / 2.0
    ax2[r, c].imshow(img, cmap="gray_r")
    ax2[r, c].axis("off")
plt.suptitle("Another set of generated images")
plt.tight_layout()
plt.show()